In [0]:
%pylab inline

In [0]:
import dataiku
from dataiku import pandasutils as pdu
import pandas as pd

In [0]:
# Example: load a DSS dataset as a Pandas dataframe
mydataset = dataiku.Dataset("mydataset")
mydataset_df = mydataset.get_dataframe()

In [0]:
import dataiku
import pandas as pd

client = dataiku.api_client()
project = client.get_project("RECIPE_USAGE")
recipes = project.list_recipes()

data_summary = []

for r_summary in recipes:
    recipe_name = r_summary['name']
    recipe_type = r_summary['type']
    
    recipe = project.get_recipe(recipe_name)
    settings = recipe.get_settings()
    raw_data = settings.data
    
    # Target the 'recipe' nested dictionary
    recipe_meta = raw_data.get('recipe', {})
    
    # --- 1. GET ORIGINAL CREATOR ---
    creation_info = recipe_meta.get('creationTag', {})
    creator = creation_info.get('lastModifiedBy', {}).get('login', 'Unknown')
    
    # --- 2. GET LAST MODIFIER ---
    version_info = recipe_meta.get('versionTag', {})
    last_modifier = version_info.get('lastModifiedBy', {}).get('login', 'Unknown')
    
    # --- ASSIGNMENT LOGIC (Based on Creator) ---
    if recipe_type == 'python':
        assigned_group = "Data scientist group"
    elif recipe_type == 'sql_query':
        assigned_group = "Data analyst group"
    else:
        assigned_group = "Other / Visual Analyst"
    
    data_summary.append({
        "Recipe Name": recipe_name,
        "Type": recipe_type,
        "Created By": creator,
        "Last Modified By": last_modifier,
        "Assigned Bucket": assigned_group
    })

# Output results
df = pd.DataFrame(data_summary)
print(df.to_string(index=False))

# If running in a Python Recipe, uncomment the lines below to save:
# output_ds = dataiku.Dataset("recipe_usage")
# output_ds.write_with_schema(df)